# Boosting, and the three libraries that implement it

This notebook picks up where the decision tree notebook left off.

There, we wrote every line ourselves and the tree was the object of study.
Here the tree is a **black box** (`sklearn`'s `DecisionTreeRegressor`), and what we build by hand is the thing *around* it: the boosting loop.

Once that loop is written, the three industrial libraries stop being magic, because each of them is that same loop with one idea added.

In [ ]:
import json
import time

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

np.random.seed(0)

## Boosting by hand
**A boosted model is a starting constant plus a list of small trees, each one fitted to what the sum of the
previous ones still gets wrong.**

$$ F_0(x) = \bar{y} \qquad\qquad F_m(x) = F_{m-1}(x) + \eta\, h_m(x) $$

where $h_m$ is a shallow regression tree fitted to the residuals of $F_{m-1}$. Two consequences of that plus
sign are worth keeping in mind while you code: nothing is ever revised (tree 3 cannot edit tree 1, only add
something on top), and every $F_m$ is available from the same fitted model, simply by stopping the sum early.

Build a 1D regression problem, the same one as in the tree notebook: $x$ uniform in $[-\pi, \pi]$ and
$y = \sin(x) + \varepsilon$ with $\varepsilon \approx \mathcal{N}(0, 0.3^2)$.

Take $200$ training points and $2000$ test points (the data is synthetic, so a large test set costs nothing
and makes every curve below readable). Keep `X` two-dimensional, `(n, 1)`, since that is what `sklearn`
expects. Plot the training set.

Do the first boosting round by hand, without any loop:

- `F = np.full(len(y), y.mean())`, the constant model $F_0$,
- `r = y - F`, the residuals it leaves,
- fit a `DecisionTreeRegressor(max_depth=2)` to `(X, r)`,
- update `F = F + eta * tree.predict(X)` with `eta = 0.1`.

Plot, side by side, the residuals `r` with the tree's prediction on top, and the data with $F_0$ and $F_1$ on
top. How much of the residual did one tree remove?

Now write the loop:
`boost(X, y, n_rounds=100, eta=0.1, max_depth=2)` returns the starting constant `F0` and the list of fitted trees,
and `predict_boost(F0, trees, X, eta, m=None)` returns $F_m(x)$, using the first `m` trees only (all of them when `m` is `None`).

Plot the residual for `n_rounds = 0, 1, 10, 50, 100`.

Print the training and the test MSE for $100$ rounds.

Plot the fit of the model for $M \in \{1, 5, 20, 100\}$ trees, in four panels, with the test MSE in each
title. Use $\eta = 0.1$ and depth-$2$ trees.

You do not need to refit anything: that is what the `m` argument of `predict_boost` is for.

### Shrinkage

$\eta$ decides how much of each tree's opinion actually makes it into the model.
Fit two models with the same $50$ trees of depth $2$, one with $\eta = 1$ and one with $\eta = 0.1$, and plot the two fits over the data.

Which one is tracking the noise?

Now measure it. For $\eta \in \{1, 0.5, 0.1, 0.05\}$, boost for $200$ rounds and record the **test** MSE
after every round (the `m` argument again, or accumulate `eta * tree.predict(X_test)` as you go).

Plot the four curves against the number of trees, and print, for each $\eta$, the lowest test MSE reached and
the number of trees at which it happened.

Check the whole thing against `sklearn.ensemble.GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=2)`:
compare its test MSE with yours, and the correlation between the two sets of predictions.

If your loop is right, the correlation is $1.0$ to five decimals.

### The same loop, for classification

Boosting a classifier changes exactly one thing: the residual.
With the log loss, the model $F$ holds **log-odds**,
the predicted probability is $p = \sigma(F)$, and the pseudo-residual of row $i$ is $y_i - p_i$.

Rebuild the two interleaved moons of the tree notebook ($200$ training points, $1000$ test points, noise $0.3$, labels in $\{0, 1\}$).
Check the generation with `plot_data`.

In [ ]:
def plot_data(X, y):
    plt.plot(X[y == 1, 0], X[y == 1, 1], 'o', color='crimson', ms=4, label='class 1')
    plt.plot(X[y == 0, 0], X[y == 0, 1], 's', color='royalblue', ms=4, label='class 0')
    plt.xlabel('$x_1$')
    plt.ylabel('$x_2$')


def plot_regions(predict, X, y, res=200, pad=0.5, title=None):
    g1 = np.linspace(X[:, 0].min() - pad, X[:, 0].max() + pad, res)
    g2 = np.linspace(X[:, 1].min() - pad, X[:, 1].max() + pad, res)
    G1, G2 = np.meshgrid(g1, g2)
    Z = np.asarray(predict(np.c_[G1.ravel(), G2.ravel()])).reshape(G1.shape)
    plt.contourf(G1, G2, Z, levels=[-0.5, 0.5, 1.5],
                 colors=['royalblue', 'crimson'], alpha=0.15)
    plot_data(X, y)
    if title:
        plt.title(title)

In [ ]:
def moons(n_per, noise=0.3):
    t = np.linspace(0, np.pi, n_per)
    X1 = np.concat([np.cos(t).reshape(1,-1), np.sin(t).reshape(1,-1)], axis=0)
    X2 = np.concat([1-np.cos(t).reshape(1,-1), 0.5-np.sin(t).reshape(1,-1)], axis=0)
    X = np.concat([X1, X2], axis=1)
    X += np.random.randn(*X.shape)*noise
    y = np.concat([np.ones_like(t), 0*np.ones_like(t)], axis=0)
    return X.T,y

X_train, y_train = moons(100)
X_test, y_test = moons(500)

Write the loop:
- `F0 = log(p0 / (1 - p0))` where `p0 = y.mean()`, the base rate as log-odds,
- at each round, `p = sigmoid(F)`, `r = y - p`, fit a `DecisionTreeRegressor(max_depth=1)` to `r`, and update
  `F += eta * tree.predict(X)`.

Run $200$ rounds at $\eta = 0.1$ and print the test accuracy.

Reminder: $\sigma(x) = \frac{1}{1 + e^{-x}}$

Plot the decision regions of the boosted classifier for $M \in \{1, 10, 50, 200\}$ trees, with the test
accuracy in each title.

-----

## XGBoost: a second order step, and a penalty inside the leaf
**Everything above used only the first derivative of the loss. XGBoost uses the second one too, and adds a ridge penalty on the leaf values.**

### The symbols

Row $i$ of the training set is $(x_i, y_i)$, and $\ell(y, F)$ is the loss, a function of the true label and of the model's output.
Everything below is written with the model of the **previous** round already fitted and frozen, and one new tree to choose.
Five symbols, three of which are just numbers:

| symbol | what it is |
|---|---|
| $F_{m-1}(x_i)$ | a **number**: the prediction the model built so far makes for row $i$, i.e. the starting constant plus the first $m-1$ trees |
| $h(x_i)$ | a **number**: the output of the new tree for row $i$. The tree $h$ itself is the unknown we are solving for |
| $g_i$ | a **number**: $\left. \dfrac{\partial \ell(y_i, F)}{\partial F} \right\rvert_{F = F_{m-1}(x_i)}$, the slope of row $i$'s loss at the current prediction |
| $h_i$ | a **number**: $\left. \dfrac{\partial^2 \ell(y_i, F)}{\partial F^2} \right\rvert_{F = F_{m-1}(x_i)}$, the curvature of the same loss at the same point |
| $w_j$ | a **number**: the value that leaf $j$ of the new tree outputs, the same for every row landing in it |

$g_i$ and $h_i$ are computable *before* the new tree exists, since they only involve the old model.
For the squared loss $\ell = \frac{1}{2}(y - F)^2$ they are $g_i = F_{m-1}(x_i) - y_i$, the residual with a sign, and $h_i = 1$;
that is why plain boosting, which knows nothing about $h_i$, still worked in the first half of this notebook.

Two hyperparameters join them: $\lambda$, a ridge penalty on each $w_j$, and $\gamma$, a fixed cost per leaf.

### Why the objective is a quadratic in $w$

The tree we are adding sits *inside* the loss, $\ell\big(y_i,\; F_{m-1}(x_i) + h(x_i)\big)$.
So expanding the loss to second order around the prediction we already have:

$$ \ell\big(y_i,\, F_{m-1}(x_i) + h(x_i)\big) \;\approx\;
   \underbrace{\ell\big(y_i,\, F_{m-1}(x_i)\big)}_{\text{contains no } h} \;+\; g_i\, h(x_i)
   \;+\; \tfrac{1}{2}\, h_i\, h(x_i)^2 $$

Two things happen there.
- The first term has no $h$ in it, so as far as choosing this tree is concerned it is a constant and can be dropped.
- And the unknown $h(x_i)$ now appears only twice, once linearly and once squared, both times multiplied by a known number.

Whatever the original loss was, what is left is a **quadratic**.

Now use what a tree actually is.
A tree is piecewise constant: every row landing in leaf $j$ receives the same output $w_j$.
So instead of summing over rows, sum over leaves and collect the rows inside each one.
All the row-level detail collapses into two numbers per leaf,

$$ G_j = \sum_{i \in \text{leaf } j} g_i \qquad\qquad H_j = \sum_{i \in \text{leaf } j} h_i $$

and, adding XGBoost's ridge penalty $\frac{1}{2}\lambda w_j^2$ on each leaf value plus a cost $\gamma$ per leaf, the whole objective becomes

$$ \sum_{j=1}^{T} \Big[\, G_j w_j + \tfrac{1}{2}\big(H_j + \lambda\big) w_j^2 \,\Big] \;+\; \gamma\, T $$

Here, we sum per tree leaf.

Minimising over the whole tree is not one $T$-dimensional problem, it is $T$ separate one-dimensional ones, each a plain quadratic in a single unknown:

$$ G w + \tfrac{1}{2}(H + \lambda)\, w^2 \qquad\Longrightarrow\qquad
   w^\star = -\frac{G}{H + \lambda} \qquad\text{and the value there is}\qquad
   -\frac{1}{2}\,\frac{G^2}{H + \lambda} $$

That second number is a **score for a set of rows**, which is exactly what we need to compare candidate splits:
cut a leaf in two, score both halves, and see whether the pair beats the whole.

$$ \text{gain} = \tfrac{1}{2}\left[ \frac{G_L^2}{H_L + \lambda} + \frac{G_R^2}{H_R + \lambda}
   - \frac{G^2}{H + \lambda} \right] - \gamma $$

Compare that with the tree notebook, where a split was scored by a *drop in impurity* computed from the labels alone.
Here the score is computed from the derivatives of the loss at the current model, so the same machinery works for any twice-differentiable loss, and the regularizer is part of the score.

Write `grad_hess_squared(y, F)` and `grad_hess_logistic(y, F)`, returning the two vectors $g$ and $h$ for the squared loss $\frac{1}{2}(y - F)^2$ and for the log loss.
In both, `F` is what the current model outputs: a prediction for the squared loss, a **log-odds** for the log loss.

```python
# squared loss: g is the residual with a sign flipped, h is always 1
g, h = grad_hess_squared(np.array([1., 2., 3.]), np.array([1., 0., 5.]))
# g -> [ 0., -2.,  2.]        g = F - y, so g = 0 for the row already predicted exactly
# h -> [ 1.,  1.,  1.]
g, h
```

```python
# log loss, a model that knows nothing: F = 0 means p = 0.5
g, h = grad_hess_logistic(np.array([1., 0.]), np.array([0., 0.]))
# g -> [-0.5,  0.5]           g = p - y
# h -> [0.25, 0.25]           h = p * (1-p); 0.25 is the largest h can ever be
g, h
```

```python
# log loss, one confident row and one confidently wrong row, both with y = 1
g, h = grad_hess_logistic(np.array([1., 1.]), np.array([4., -4.]))
# p -> [0.982, 0.018]
# g -> [-0.018, -0.982]
# h -> [ 0.018,  0.018]
g, h
```

Write `leaf_value(g, h, lam)` returning $w^\star$, and `split_gain(g, h, mask, lam, gamma=0.0)` returning the gain of the split that sends the rows of `mask` left.

```python
leaf_value(np.array([-2.]), np.array([1.]), lam=0.0),\
leaf_value(np.array([-2.]), np.array([1.]), lam=5.0)   # -> 2.0, 0.333
```

```python
g, h = np.array([-1., -3.]), np.ones(2)
leaf_value(g, h, lam=0.0),\
leaf_value(g, h, lam=2.0)  # -> 2.0, 1.0
```

```python
g, h = np.array([-1., -1., 1., 1.]), np.ones(4)
split_gain(g, h, np.array([True, False, True, False]), lam=0.0),\
split_gain(g, h, np.array([True, True, False, False]), lam=0.0),\
split_gain(g, h, np.array([True, True, False, False]), lam=2.0),\
split_gain(g, h, np.array([True, True, False, False]), lam=0.0, gamma=2.0)   # -> 0.0, 2.0, 1.0, 0.0
```

Now the check that this really is XGBoost.
Fit a single stump on the wave data with every source of difference switched off:
`max_depth=1` to give exactly one split,
`base_score=0.0` to make the current model predict $0$ for every row,
`eta=1.0` to stop the leaf values from being shrunk before they are written into the dump,
`min_child_weight=0` to remove XGBoost's floor on $H$ per leaf, and
`tree_method='exact'` to stop it from binning the column.

**Recompute those two leaf values.**
1. the model before this tree predicts `base_score` for every row, so `F = np.zeros(len(y_train))`,
2. `g, h = grad_hess_squared(y_train, F)`, which for this loss is `g = -y_train` and `h = 1`,
3. `left = X_train[:, feature] < threshold`, the rows XGBoost sends to the `yes` child. **The comparison is
   strict `<`, not `<=`**, which is the opposite of the convention you used in the tree notebook,
4. `leaf_value(g[left], h[left], lam=5.0)` and the same on `~left`, with `lam` equal to the `reg_lambda` you
   passed.

Print the two pairs of numbers next to each other.

In [ ]:
d = xgb.DMatrix(X_train, label=y_train)
params = {'objective': 'reg:squarederror', 'max_depth': 1, 'eta': 1.0, 'reg_lambda': 5.0, 'base_score': 0.0, 'min_child_weight': 0, 'tree_method': 'exact'}
bst = xgb.train(params, d, num_boost_round=1)
print(bst.get_dump(dump_format='json')[0])

In [ ]:
node = json.loads(bst.get_dump(dump_format='json')[0])

feature   = int(node['split'][1:])       # 'f0' -> column 0
threshold = node['split_condition']
leaf      = {child['nodeid']: child['leaf'] for child in node['children']}
xgb_left, xgb_right = leaf[node['yes']], leaf[node['no']]


Do the same for classification: the identical experiment with `objective='binary:logistic'` and `base_score=0.5`, recomputing the two leaf values with `grad_hess_logistic`.

Plot $w^\star$ against $\lambda$ for one of those two leaves, for $\lambda$ on a log grid from $10^{-2}$ to
$10^{3}$. Add a horizontal line at the unpenalised value $-G/H$.

Nothing needs to be refitted: the leaf you just checked is a fixed set of rows, and those rows give a fixed
pair of numbers. Reuse `g`, `h` and `left` from the cell above and collapse them with
`G, H = g[left].sum(), h[left].sum()`. Then $w^\star = -G/(H+\lambda)$ is a one-liner in $\lambda$, and
`np.logspace(-2, 3, 200)` gives the grid (`logspace` takes the *exponents*, so those are $10^{-2}$ to
$10^{3}$). Use `plt.semilogx` so the grid is evenly spaced on screen, and `plt.axhline(-G/H)` for the
reference value.

Two things to look for in the curve. On the left, where $\lambda \ll H$, it is flat and sits on the
horizontal line: the penalty is invisible next to the total Hessian of the leaf. On the right, where
$\lambda \gg H$, $w^\star \approx -G/\lambda \to 0$, and on a log $x$ axis that tail is a slow decay, not a
cliff. The crossover happens at $\lambda \approx H$, which for the squared loss is just the number of rows in
the leaf, so the *same* $\lambda$ shrinks a small leaf hard and a large leaf barely at all. That is the whole
reason `reg_lambda` acts as a regulariser rather than a uniform rescaling.

This is the same picture as ridge regression's coefficient path, one level down.

Here is a code that generates a bigger tabular problem (generate $10\,000$ rows and $20$ columns carrying a signal of decaying strength):

In [ ]:
def make_data(n):
    W = 0.9 ** np.arange(20)
    X = np.random.randn(n, 20)
    y = (X[:, :20] @ W + np.random.randn(n) * 0.5 > 0).astype(int)
    return X, y

X_train, y_train = make_data(10_000)
X_eval,  y_eval  = make_data(1_000)
X_test,  y_test  = make_data(1_000)

And here is a code that fits `xgb.XGBClassifier` and plots the validation log loss:

In [ ]:
model = xgb.XGBClassifier(n_estimators=2000, learning_rate=0.05,
                          early_stopping_rounds=50, eval_metric='logloss')
model.fit(X_train, y_train, eval_set=[(X_eval, y_eval)], verbose=False)
print(model.best_iteration)
plt.axvline(model.best_iteration, color='grey')
plt.semilogy(model.evals_result_['validation_0']['logloss'])
plt.show()

Modify the learning rate ($\times 10$, $\frac{.}{10}$) and refit.
What happens to `best_iteration`, and to the validation score?

In [ ]:
model = xgb.XGBClassifier(n_estimators=2000, learning_rate=0.005,
                          early_stopping_rounds=50, eval_metric='logloss')
model.fit(X_train, y_train, eval_set=[(X_eval, y_eval)], verbose=False)
plt.axvline(model.best_iteration, color='grey')
plt.semilogy(model.evals_result_['validation_0']['logloss'])
plt.show()


-----

## LightGBM: stop looking at rows
**Your `best_split` in the tree notebook tried every midpoint between two sorted values, which is $n - 1$
candidates per column, and it recomputed the impurity of the children each time. LightGBM makes both of those
costs disappear.**

**Bin once, then forget the rows.**
The code below takes one column of the tabular problem above, cuts it into `n_bins` quantile bins with `np.quantile` and `np.searchsorted`, prints the number of candidate
thresholds each scan would consider, and plots the bin assignments.

Run it, then set `n_bins = 16` and run it again.

In [ ]:
n_bins = 255
col = X_train[:, 0]
edges = np.quantile(col, np.linspace(0, 1, n_bins + 1)[1:-1])
b = np.searchsorted(edges, col)          # bin index of every row, 0 to n_bins-1

print(f'exact scan:  {len(col) - 1} candidate thresholds')
print(f'binned scan: {n_bins} candidate thresholds')

plt.hist(b, bins=n_bins)
plt.xlabel('bin index')
plt.ylabel('rows in the bin')
plt.show()

**A child's histogram is the parent's minus its sibling's.**
Once the rows are binned, a split is just a partition of the bin counts, and the two children are not independent: whatever does not go left goes right.
The code below checks that numerically on an arbitrary split of the rows.

Run it, then change the threshold in `mask` to something lopsided, say `X_train[:, 1] < -1.5`, and run it again.

In [ ]:
mask = X_train[:, 1] < 0

parent      = np.bincount(b,        minlength=n_bins)
child_left  = np.bincount(b[mask],  minlength=n_bins)
child_right = np.bincount(b[~mask], minlength=n_bins)

print('parent - left == right:', np.array_equal(parent - child_left, child_right))
print(f'rows on the left {mask.sum()}, rows on the right {(~mask).sum()}')

**Measure what binning buys.**
The code below generates a dataset, keeps $80\%$ of it for training, and times three fits of $100$ trees of depth $6$:
XGBoost scanning every candidate threshold (`tree_method='exact'`), XGBoost on binned columns (`tree_method='hist'`, its default since version $1.7$), and LightGBM.

Run it, then call `time_three(5_000)` instead.

In [ ]:
def time_three(n):
    X, y = make_data(n)
    cut = n * 4 // 5
    X_tr, y_tr, X_va, y_va = X[:cut], y[:cut], X[cut:], y[cut:]
    models = [
        ('xgb exact', xgb.XGBClassifier(n_estimators=100, max_depth=6, tree_method='exact')),
        ('xgb hist',  xgb.XGBClassifier(n_estimators=100, max_depth=6, tree_method='hist')),
        ('lightgbm',  lgb.LGBMClassifier(n_estimators=100, max_depth=6, verbose=-1)),
    ]
    for name, model in models:
        t0 = time.time()
        model.fit(X_tr, y_tr)
        print(f'{name:10s} {time.time() - t0:6.2f} s   accuracy {model.score(X_va, y_va):.3f}')

time_three(100_000)

**Leaf-wise growth.**
LightGBM does not grow level by level: at each step it splits the leaf that promises the largest gain, wherever it sits in the tree.
So its capacity knob is `num_leaves`, not `max_depth`.

The code below fits LightGBM on a *small* training set for five values of `num_leaves` and prints the training and validation accuracy of each.
(`min_child_samples=1` removes LightGBM's default floor of $20$ rows per leaf, which would otherwise hide the effect on $1000$ rows.)

Run it, then set `n_small = 20_000` and run it again.

In [ ]:
n_small = 1_000
X_small, y_small = make_data(n_small)
X_val, y_val = make_data(1_000)

for num_leaves in [4, 8, 31, 127, 511]:
    model = lgb.LGBMClassifier(n_estimators=100, num_leaves=num_leaves,
                               min_child_samples=1, verbose=-1)
    model.fit(X_small, y_small)
    print(f'num_leaves {num_leaves:4d}   train {model.score(X_small, y_small):.3f}'
          f'   validation {model.score(X_val, y_val):.3f}')

-----

## CatBoost: categories, without leaking the target
**A category is not a number, and the obvious way to make it one quietly copies the answer into the
features.**

The code below generates $3000$ training rows and $3000$ test rows with three ordinary numeric columns,
a label that depends only on the first one, and one categorical column carrying **no information at all**:
a random integer id drawn from `n_categories` values (think customer id, or sample barcode).

It encodes that column the obvious way, by its **target statistic** (each category replaced by the mean of `y` over the training rows in that category), and fits a `DecisionTreeClassifier(max_depth=8)` with and without it.

Run it, then set `n_categories = 30` and run it again.

In [ ]:
def make_cat_data(n, n_categories):
    X = np.random.randn(n, 3)
    y = (X[:, 0] * 1.5 + np.random.randn(n) > 0).astype(int)
    cat = np.random.randint(0, n_categories, n)
    return X, cat, y

n_categories = 3_000
X_tr, cat_tr, y_tr = make_cat_data(3_000, n_categories)
X_te, cat_te, y_te = make_cat_data(3_000, n_categories)

# target statistic, computed on the training rows
p = y_tr.mean()                                                   # prior, the global mean of y
sums = np.bincount(cat_tr, weights=y_tr, minlength=n_categories)
counts = np.bincount(cat_tr, minlength=n_categories)
encoding = np.where(counts > 0, sums / np.maximum(counts, 1), p)

Xe_tr = np.column_stack([X_tr, encoding[cat_tr]])
Xe_te = np.column_stack([X_te, encoding[cat_te]])

tree = DecisionTreeClassifier(max_depth=8).fit(Xe_tr, y_tr)
print(f'numeric + target statistic   train {tree.score(Xe_tr, y_tr):.3f}   test {tree.score(Xe_te, y_te):.3f}')
tree = DecisionTreeClassifier(max_depth=8).fit(X_tr, y_tr)
print(f'numeric only                 train {tree.score(X_tr, y_tr):.3f}   test {tree.score(X_te, y_te):.3f}')

**Ordered target statistics.**

The fix is to compute each row's encoding from rows that came *before* it in a random permutation, never from the row itself:

$$ \hat{x}_i = \frac{\sum_{j \prec i,\; c_j = c_i} y_j + a\, p}{\#\{ j \prec i,\; c_j = c_i \} + a} $$

with $p$ the global mean of `y` (the prior) and $a$ a smoothing weight.

The code below is that formula, as a single pass over a random permutation keeping a running sum and count per category.
The **test** rows have no "before", so they get the full training statistic with the same prior smoothing.

Run it (with `n_categories = 3_000` again in the cell above), then set `a = 100.0` and run it again.

In [ ]:
a = 1.0                                    # smoothing weight
perm = np.random.permutation(len(y_tr))

running_sum, running_count = np.zeros(n_categories), np.zeros(n_categories)
ordered = np.empty(len(y_tr))
for i in perm:
    c = cat_tr[i]
    ordered[i] = (running_sum[c] + a * p) / (running_count[c] + a)   # before the row is added
    running_sum[c] += y_tr[i]
    running_count[c] += 1

Xo_tr = np.column_stack([X_tr, ordered])
Xo_te = np.column_stack([X_te, (sums[cat_te] + a * p) / (counts[cat_te] + a)])

tree = DecisionTreeClassifier(max_depth=8).fit(Xo_tr, y_tr)
print(f'numeric + ordered statistic  train {tree.score(Xo_tr, y_tr):.3f}   test {tree.score(Xo_te, y_te):.3f}')

**The library.**

CatBoost does all of that for you, on a `pandas.DataFrame` whose categorical columns are declared in `cat_features` and held as **strings**.

Run the code below, then set `one_hot_max_size = 5_000`, which is above `n_categories` and therefore tells CatBoost to one-hot encode the column instead of using ordered statistics on it.

Compare the four test accuracies of the section.

And since one-hot encoding never looks at `y` at all, it cannot leak the target, so say instead what it costs on a column with $3000$ values, and why that cost is the reason target statistics were invented.

In [ ]:
def frame(X, cat):
    return pd.DataFrame({'x1': X[:, 0], 'x2': X[:, 1], 'x3': X[:, 2], 'cat': cat.astype(str)})

df_tr, df_te = frame(X_tr, cat_tr), frame(X_te, cat_te)

one_hot_max_size = 2                       # the default, so the column goes through ordered statistics
model = CatBoostClassifier(iterations=200, depth=6, learning_rate=0.1, cat_features=['cat'],
                           one_hot_max_size=one_hot_max_size, verbose=0, random_seed=0,
                           allow_writing_files=False)
model.fit(df_tr, y_tr)
print(f'catboost                     train {model.score(df_tr, y_tr):.3f}   test {model.score(df_te, y_te):.3f}')

-----

## Choosing

One dataset ($50\,000$ rows and $20$ columns from the same generator, split into train, validation and test), one protocol:
early stopping on the validation set after $50$ rounds without improvement, and the same learning rate for all three libraries.

The code below runs it and prints, for each library, the number of trees early stopping chose, the fit time and the test accuracy.

Run it, then set `learning_rate = 0.5` and run it again.

In [ ]:
X, y = make_data(50_000)
X_tr, y_tr = X[:30_000], y[:30_000]
X_va, y_va = X[30_000:40_000], y[30_000:40_000]
X_te, y_te = X[40_000:], y[40_000:]

learning_rate = 0.05
models = {}

for name in ('xgboost', 'lightgbm', 'catboost'):
    t0 = time.time()
    match name:
        case 'xgboost':
            model = xgb.XGBClassifier(n_estimators=2000, learning_rate=learning_rate,
                                      early_stopping_rounds=50, eval_metric='logloss')
            model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
            n_trees = model.best_iteration
        case 'lightgbm':
            model = lgb.LGBMClassifier(n_estimators=2000, learning_rate=learning_rate, verbose=-1)
            model.fit(X_tr, y_tr, eval_X=X_va, eval_y=y_va,
                      callbacks=[lgb.early_stopping(50, verbose=False)])
            n_trees = model.best_iteration_
        case 'catboost':
            model = CatBoostClassifier(iterations=2000, learning_rate=learning_rate,
                                       early_stopping_rounds=50, verbose=0,
                                       random_seed=0, allow_writing_files=False)
            model.fit(X_tr, y_tr, eval_set=(X_va, y_va))
            n_trees = model.get_best_iteration()
    fit_time = time.time() - t0
    models[name] = model
    print(f'{name:10s} {n_trees:5d} trees   {fit_time:6.2f} s   test accuracy {model.score(X_te, y_te):.4f}')

The code below plots the `feature_importances_` of the three fitted models side by side, as bar charts over the $20$ columns, each normalised to sum to one so that the three panels are on the same scale.

Run it, then set `importance_type = 'count'` and run it again.

In [ ]:
importance_type = 'gain'
xgb_type, lgb_type = {'gain': ('gain', 'gain'), 'count': ('weight', 'split')}[importance_type]
models['xgboost'].importance_type = xgb_type
models['lightgbm'].importance_type = lgb_type

fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for ax, (name, model) in zip(axes, models.items()):
    importance = model.feature_importances_
    ax.bar(np.arange(20), importance / importance.sum())
    ax.set_title(name)
    ax.set_xlabel('column')
axes[0].set_ylabel('share of total importance')
plt.tight_layout()
plt.show()